# Análise do CICDDoS2019 01/12

## Criação do Dataset Balanceado

In [ ]:
from src.Data.Handler import DatasetHandler
from pathlib import Path

RAW_DATASET_DIR = Path("datasets") / "CICDDoS2019" / "01-12"

handler = DatasetHandler(logging=True)

handler.create_balanced_dataset(
    src_dir=RAW_DATASET_DIR, 
    dest_dir="data", 
    output_filename="BALANCED.CSV", 
    n_samples_per_class=10000,
    chunk_size=100000,
    target_files=None,
    ignored_classes=['WebDDoS'],
    allow_insufficient=True
)

## Análise de Similaridade

In [ ]:
import pandas as pd
from src.Data.Processor import DataStreamProcessor
from src.Data.Handler import DatasetHandler

processor = DataStreamProcessor(logging=True)
handler = DatasetHandler(logging=True)

# Carrega o CSV 
df_bruto = pd.read_csv('data/BALANCED.CSV')

# Pré-Processamento 
X_df, y_array, target_names = processor.create_stream(
    df=df_bruto,
    target_label_col='Label',
    binary_label=False,
    normalize_method='StandardScaler',
    imputation_method='median',
    extra_ignore_cols=['Source IP', 'Source Port','Destination IP','Destination Port','Protocol', 'Inbound'],
    return_stream=False
)

# Executa a Seleção OvA (One-vs-All)
dict_features, vetor_features_finais = handler.extract_ovr_feature_importance(
    X=X_df, 
    y=y_array, 
    target_names=target_names,
    top_per_class=10
)

In [ ]:
%load_ext autoreload
%autoreload 2

# Plota os Dendrogramas e Mapas de Calor
handler.plot_similarity_and_feature_groups(
    X=X_df, 
    y=y_array, 
    target_names=target_names,
    selected_features=vetor_features_finais, 
    metric='euclidean',
    linkage_method='ward'
)

# Criação de cenários

## Média de Amostras Normais 15k

In [ ]:
from pathlib import Path

RAW_DATASET_DIR = Path("datasets") / "CICDDoS2019" / "01-12"

AMOSTRAS = [25, 200, 1000]
BASELINE = "data/BENIGN.csv"
INPUT_FOLDER = RAW_DATASET_DIR
OUTPUT_FOLDER = "data/15k"

### Cenário de Consistência

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Consistência/Consistência_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])

### Cenário Generalização

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras),
        ("DrDoS_LDAP.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Generalização/Generalização_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])

### Cenário Adaptação

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras),
        ("Syn.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Adaptação/Adaptação_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])

### Cenário de Recorrência

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras//2),
        ("Syn.csv", qtd_amostras//2),
        ("DrDoS_LDAP.csv", qtd_amostras//2),
        ("DrDoS_DNS.csv", qtd_amostras//2),
        ("Syn.csv", qtd_amostras//2),
        ("DrDoS_LDAP.csv", qtd_amostras//2)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Recorrência/Recorrência_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])